In [0]:
display(dbutils.fs.ls('abfss://rag-logs@rahulchurndatalake.dfs.core.windows.net/logs'))

In [0]:
from pyspark.sql import SparkSession
from datetime import datetime, timedelta
import random

spark = SparkSession.builder.getOrCreate()

jobs = ["Ingestion", "Transformation", "Load", "Validation"]
statuses = ["SUCCESS", "FAILED"]
errors = [
    "Schema mismatch",
    "Null value violation",
    "Permission denied",
    "Timeout error",
    "File not found",
    None
]

data = []

base_time = datetime(2026, 4, 20, 10, 0, 0)

for i in range(200):
    job = random.choice(jobs)
    status = random.choice(statuses)
    error = random.choice(errors) if status == "FAILED" else None
    
    timestamp = base_time + timedelta(minutes=i*5)

    data.append((
        timestamp.strftime("%Y-%m-%d %H:%M:%S"),
        job,
        status,
        error
    ))

df = spark.createDataFrame(data, ["timestamp", "job_name", "status", "error_message"])

display(df)

In [0]:
output_path = "abfss://rag-logs@rahulchurndatalake.dfs.core.windows.net/logs/"
df.write.mode("overwrite").option("header", "true").csv(output_path)

In [0]:
%sql
CREATE TABLE rag_demo.logs.raw_logs
USING CSV
OPTIONS (
  path 'abfss://rag-logs@rahulchurndatalake.dfs.core.windows.net/logs/',
  header 'true'
);

In [0]:
%sql
select * from rag_demo.logs.raw_logs